In [ ]:
# Step 2: Temporal stability mask for DEA Land Cover transitions

In [ ]:
# ===============================================================
# Step-2 — Stability check for DEA Land Cover transitions
# Windows: fixed cadence (e.g., 5-yr) + custom hydrological epochs
# Period: 1988–2023
# Outputs:
#   1) QC fields (valid/kept pixels, % kept, min/max votes)
#   2) Stability-filtered woody/encroachment/retreat stats
# ===============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
from datacube.utils.geometry import Geometry, CRS
from shapely.geometry import mapping
import datacube
from datetime import datetime

# -------------------
# CONFIG
# -------------------
START_YEAR   = 1988
END_YEAR     = 2023

# Fixed cadence (primary). Re-run with 10 if needed.
INTERVAL_YRS = 5

# Hydrological epochs (EDIT as needed)
CUSTOM_EPOCHS = [
    {"label": "1988-1993_WetBaseline",        "start": 1988, "end": 1993},
    {"label": "1994-2000_EarlyDry",           "start": 1994, "end": 2000},
    {"label": "2001-2009_MillenniumDrought",  "start": 2001, "end": 2009},
    {"label": "2010-2012_PostDroughtFloods",  "start": 2010, "end": 2012},
    {"label": "2013-2017_ModerateDry",        "start": 2013, "end": 2017},
    {"label": "2018-2019_SevereDrought",      "start": 2018, "end": 2019},
    {"label": "2020-2022_LaNinaFloods",       "start": 2020, "end": 2022},
    # 2023-2023 would be zero-length; omit or set end=2024 if you really want it.
]

PIXEL_RES      = 30
AREA_PER_PX_HA = (PIXEL_RES**2) / 1e4
TARGET_CRS     = "EPSG:3577"
LC_PRODUCT     = "ga_ls_landcover_class_cyear_3"
BUFFER_M       = 500

# Stability parameters (majority vote over ±window)
STABILITY_WINDOW = 1      # years on each side (e.g., 1 => 3-year window)
MIN_VALID_VOTES  = 2      # minimum valid-year votes required

BASE_DIR   = "/home/jovyan/DEA products paper/Data"
TABLE_DIR2 = "/home/jovyan/DEA products paper/Results/step 2_v3/tables"
os.makedirs(TABLE_DIR2, exist_ok=True)

# Wetland shapefiles (same as Step-1)
WETLAND_SHP_PATHS = [
    os.path.join(BASE_DIR, "wetland boundaries/P1ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P2ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P3ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P4ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P5ANAEWetlands.shp"),
]

# -------------------
# CLASS GROUPS (Natural-only + ambiguous NAT kept)
# -------------------
WOODY_CODES = {20,27,28,29,30,31, 56,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77}
HERB_CODES  = {21,32,33,34,35,36, 57,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92}
BARE_CODES  = {94,95,96,97}
WATER_CODES = {98,99,100,101,102,103,104}

# Ambiguous NATURAL (included as enc source + supplementary retreat)
AMBIG_NAT   = {19,22,23,24,25,26, 55,58,59,60,61,62}

# Exclusions (kept OUT of validity)
ARTIFICIAL_CODES = {93}
NODATA_CODES     = {255}
CULTIVATED       = {1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18}

MASK_OUT_CODES   = ARTIFICIAL_CODES | NODATA_CODES | CULTIVATED

# -------------------
# HELPERS
# -------------------
def make_periods(start, end, interval_years=None, custom_epochs=None):
    """
    Returns list of (label, y0, y1).
    - interval_years: rolling baseline->target windows (e.g., 5 or 10)
    - custom_epochs: list of dicts {"label": str, "start": int, "end": int}
    """
    periods = []
    if interval_years:
        years = list(range(start, end, interval_years))
        for y0 in years:
            y1 = y0 + interval_years
            if y1 <= end:
                periods.append((f"{y0}-{y1}", y0, y1))
    if custom_epochs:
        for ep in custom_epochs:
            label = ep["label"]
            y0, y1 = int(ep["start"]), int(ep["end"])
            if y0 < y1 and (start <= y0 < y1 <= end):
                periods.append((label, y0, y1))
    return periods

def dc_query_from_geom(geom, crs_str=TARGET_CRS, res=PIXEL_RES, buffer_m=BUFFER_M):
    geom_buf = geom.buffer(buffer_m)
    return {"geopolygon": Geometry(mapping(geom_buf), CRS(crs_str)),
            "output_crs": crs_str, "resolution": (-res, res)}

def mask_codes(da: xr.DataArray, keep: set) -> xr.DataArray:
    codes = np.array(list(keep), dtype="int32")
    return xr.apply_ufunc(np.isin, da.astype("int32"), codes,
                          dask="parallelized", output_dtypes=[bool])

def count_px(mask: xr.DataArray) -> int:
    return int(mask.sum().values)

def ha(mask: xr.DataArray) -> float:
    return float(mask.sum().values) * AREA_PER_PX_HA

def pct(part, whole):
    return (part / whole * 100.0) if (whole and whole > 0) else np.nan

def get_year_slice(da: xr.DataArray, year: int):
    if "time" not in da.dims:
        return None
    subset = da.sel(time=da.time.dt.year == year)
    if subset.sizes.get("time", 0) == 0:
        return None
    return subset.squeeze(drop=True)

def window_years(y, lo=START_YEAR, hi=END_YEAR, w=STABILITY_WINDOW):
    return [yy for yy in range(y-w, y+w+1) if lo <= yy <= hi]

def woody_majority_stable(da_stack: xr.DataArray, focus_year: int):
    """
    Majority-vote stability around focus_year.
    Returns (stable_mask, valid_votes) as DataArrays aligned to spatial dims.
    A pixel is 'stable' if, given MIN_VALID_VOTES or more valid observations in the window:
      - it is woody in focus_year and woody_votes >= majority threshold, OR
      - it is non-woody in focus_year and woody_votes < majority threshold.
    Validity excludes MASK_OUT_CODES at each year in the window.
    """
    years = window_years(focus_year)
    ref = da_stack.sel(time=da_stack.time.dt.year == focus_year)
    if ref.size == 0:
        return xr.DataArray(np.array([])), xr.DataArray(np.array([]))
    ref = ref.squeeze(drop=True)

    if len(years) == 0:
        return xr.zeros_like(ref, dtype=bool), xr.zeros_like(ref, dtype="int16")

    woody_list, valid_list = [], []
    for yy in years:
        sl = da_stack.sel(time=da_stack.time.dt.year == yy)
        if sl.size == 0:
            continue
        sl = sl.squeeze(drop=True)
        valid = ~mask_codes(sl, MASK_OUT_CODES)
        woody = mask_codes(sl, WOODY_CODES) & valid
        woody_list.append(woody.astype("int16"))
        valid_list.append(valid.astype("int16"))

    if not woody_list:
        return xr.zeros_like(ref, dtype=bool), xr.zeros_like(ref, dtype="int16")

    valid_votes = sum(valid_list)
    woody_votes = sum(woody_list)
    thr = (valid_votes + 1)//2

    sl_focus = ref
    valid_focus = ~mask_codes(sl_focus, MASK_OUT_CODES)
    woody_focus = mask_codes(sl_focus, WOODY_CODES) & valid_focus

    stable = xr.where(
        (valid_votes >= MIN_VALID_VOTES) &
        (((woody_focus) & (woody_votes >= thr)) |
         ((~woody_focus) & valid_focus & (woody_votes < thr))),
        True, False
    )
    return stable, valid_votes

def safe_vote_range(votes_da: xr.DataArray) -> str:
    try:
        if votes_da.size == 0:
            return "0–0"
        vals = votes_da.values
        if vals.size == 0:
            return "0–0"
        vmin = np.nanmin(vals); vmax = np.nanmax(vals)
        if np.isnan(vmin) or np.isnan(vmax):
            return "0–0"
        return f"{int(vmin)}–{int(vmax)}"
    except Exception:
        return "0–0"

# -------------------
# LOAD WETLANDS
# -------------------
wetland_list = []
for sub_idx, path in enumerate(WETLAND_SHP_PATHS, start=1):
    gdf = gpd.read_file(path).to_crs(TARGET_CRS)
    if "OBJECTID" not in gdf.columns:
        raise ValueError(f"{path} has no OBJECTID field!")
    gdf["WetlandID"] = gdf["OBJECTID"].astype(int)
    gdf["SubRegion"] = f"SR{sub_idx}"
    gdf["Area_ha"] = gdf.geometry.area/10000.0
    wetland_list.append(gdf)
wetlands = gpd.GeoDataFrame(pd.concat(wetland_list, ignore_index=True), crs=TARGET_CRS)

# -------------------
# MAIN
# -------------------
dc = datacube.Datacube(app="WPE_LC_L4_Stability")
rows = []

all_periods = make_periods(
    START_YEAR, END_YEAR,
    interval_years=INTERVAL_YRS,
    custom_epochs=CUSTOM_EPOCHS
)
print("Total periods (fixed + epochs):", len(all_periods))

for subr in wetlands["SubRegion"].unique():
    sub_wetlands = wetlands[wetlands["SubRegion"] == subr].reset_index(drop=True)
    n_total = len(sub_wetlands)
    print(f"▶ Processing {subr} ... ({n_total} wetlands)")

    for idx, wr in sub_wetlands.iterrows():
        if (idx == 0) or ((idx+1) % 100 == 0) or (idx+1 == n_total):
            print(f"   ▶ Wetland {idx+1}/{n_total} in {subr}")

        wid, geom = int(wr["WetlandID"]), wr.geometry
        area_ha = float(wr["Area_ha"])

        q = dc_query_from_geom(geom)
        ds = dc.load(
            product=LC_PRODUCT,
            measurements=["full_classification"],
            time=(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"),
            **q
        )
        if ds.sizes.get("time",0) == 0:
            continue

        try:
            da_stack = ds["full_classification"].rio.clip([geom], crs=TARGET_CRS, drop=True)
        except Exception:
            print(f"⚠️ Wetland {wid} in {subr} skipped (no overlap).")
            continue

        for label, y0, y1 in all_periods:
            base = get_year_slice(da_stack, y0)
            tgt  = get_year_slice(da_stack, y1)
            if base is None or tgt is None:
                continue

            # Stability masks at endpoints
            base_stable, base_votes = woody_majority_stable(da_stack, y0)
            tgt_stable,  tgt_votes  = woody_majority_stable(da_stack, y1)
            if base_stable.size == 0 or tgt_stable.size == 0:
                continue
            stable_mask = base_stable & tgt_stable

            # Validity (exclude cultivated/urban/NoData) at endpoints
            valid_base = ~mask_codes(base, MASK_OUT_CODES)
            valid_tgt  = ~mask_codes(tgt,  MASK_OUT_CODES)
            valid = valid_base & valid_tgt

            # QC before/after stability filtering
            valid_px_before = count_px(valid)
            kept_px_after   = count_px(valid & stable_mask)
            kept_pct        = (kept_px_after/valid_px_before*100.0) if valid_px_before>0 else np.nan
            base_vote_range = safe_vote_range(base_votes)
            tgt_vote_range  = safe_vote_range(tgt_votes)

            # Apply stability
            valid = valid & stable_mask

            # Group masks (natural-only; ambiguous NAT kept)
            base_woody  = mask_codes(base, WOODY_CODES)  & valid
            tgt_woody   = mask_codes(tgt,  WOODY_CODES)  & valid

            base_herb   = mask_codes(base, HERB_CODES)   & valid
            base_bare   = mask_codes(base, BARE_CODES)   & valid
            base_water  = mask_codes(base, WATER_CODES)  & valid
            base_ambig  = mask_codes(base, AMBIG_NAT)    & valid

            # Encroachment (include ambiguous→woody)
            enc_herb    = base_herb   & tgt_woody
            enc_bare    = base_bare   & tgt_woody
            enc_water   = base_water  & tgt_woody
            enc_ambig   = base_ambig  & tgt_woody
            enc_any     = enc_herb | enc_bare | enc_water | enc_ambig

            # Retreat (clear) — woody → herb/bare/water
            ret_herb    = base_woody & mask_codes(tgt, HERB_CODES)  & valid
            ret_bare    = base_woody & mask_codes(tgt, BARE_CODES)  & valid
            ret_water   = base_woody & mask_codes(tgt, WATER_CODES) & valid
            ret_any_clear = ret_herb | ret_bare | ret_water

            # Retreat (supplementary) — woody → ambiguous NAT
            ret_ambig   = base_woody & mask_codes(tgt, AMBIG_NAT)   & valid
            ret_any_ext = ret_any_clear | ret_ambig

            # Areas (ha)
            trees_base_ha = ha(base_woody)
            trees_tgt_ha  = ha(tgt_woody)

            enc_herb_ha   = ha(enc_herb)
            enc_bare_ha   = ha(enc_bare)
            enc_water_ha  = ha(enc_water)
            enc_ambig_ha  = ha(enc_ambig)
            enc_any_ha    = ha(enc_any)

            ret_h2h_ha       = ha(ret_herb)
            ret_w2b_ha       = ha(ret_bare)
            ret_w2w_ha       = ha(ret_water)
            ret_any_clear_ha = ha(ret_any_clear)

            ret_w2a_ha       = ha(ret_ambig)
            ret_any_ext_ha   = ha(ret_any_ext)

            d_trees_ha = trees_tgt_ha - trees_base_ha
            pct_trees  = pct(d_trees_ha, trees_base_ha)

            years_len = max(1, (y1 - y0))
            # Annualised rates (ha/yr)
            enc_any_ha_peryr      = enc_any_ha / years_len
            ret_clear_ha_peryr    = ret_any_clear_ha / years_len
            ret_extended_ha_peryr = ret_any_ext_ha / years_len
            d_trees_ha_peryr      = d_trees_ha / years_len

            rows.append({
                "WetlandID": wid, "SubRegion": subr,
                "Period": label, "BaselineYear": y0, "TargetYear": y1,
                "YearsLen": years_len, "Area_ha": area_ha,
                # QC
                "Valid_px_before": valid_px_before,
                "Kept_px_after":   kept_px_after,
                "Kept_%of_valid":  kept_pct,
                "Base_min_max_votes": base_vote_range,
                "Tgt_min_max_votes":  tgt_vote_range,
                # Woody extents
                "Trees_Base_ha": trees_base_ha,
                "Trees_Target_ha": trees_tgt_ha,
                "ΔTrees_ha": d_trees_ha,
                "%ΔTrees": pct_trees,
                "ΔTrees_ha_peryr": d_trees_ha_peryr,
                # Encroachment (stability-filtered)
                "Enc_Herb_to_Woody_ha_stab":   enc_herb_ha,
                "Enc_Bare_to_Woody_ha_stab":   enc_bare_ha,
                "Enc_Water_to_Woody_ha_stab":  enc_water_ha,
                "Enc_Ambig_to_Woody_ha_stab":  enc_ambig_ha,
                "Enc_Any_to_Woody_ha_stab":    enc_any_ha,
                "Enc_Any_to_Woody_ha_peryr_stab": enc_any_ha_peryr,
                # Retreat (stability-filtered)
                "Ret_Woody_to_Herb_ha_stab":   ret_h2h_ha,
                "Ret_Woody_to_Bare_ha_stab":   ret_w2b_ha,
                "Ret_Woody_to_Water_ha_stab":  ret_w2w_ha,
                "Ret_Woody_to_Ambig_ha_stab":  ret_w2a_ha,              # supplementary
                "Ret_Any_to_NonWoody_ha_stab": ret_any_clear_ha,        # clear only
                "Ret_Any_to_NonWoody_Extended_ha_stab": ret_any_ext_ha, # clear + ambiguous
                "Ret_Any_to_NonWoody_ha_peryr_stab": ret_clear_ha_peryr,
                "Ret_Any_to_NonWoody_Extended_ha_peryr_stab": ret_extended_ha_peryr,
            })

# -------------------
# EXPORT
# -------------------
stab_df = pd.DataFrame(rows).sort_values(
    ["SubRegion","WetlandID","BaselineYear","TargetYear"]
).reset_index(drop=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(TABLE_DIR2,
    f"stability_summary_{START_YEAR}_{END_YEAR}_{INTERVAL_YRS}yr_plusEpochs_{ts}.csv")
stab_df.to_csv(csv_path, index=False)

print("✅ Exported stability summary to:", csv_path)
stab_df.head()